# Retake Coach: run the demo page

**Cost: $0 for detection** (DINOv2 runs here). Muse Spark wording is capped in the last cell: $0.25 a day, 60 requests a day,
and $1.00 in total for this runtime. Set `USE_MUSE = False` to spend nothing; the page then uses standard instructions.
**Runtime:** GPU is fastest (T4 is enough). CPU works too, at about 1–2 seconds a photo.

Needs `quality/coach_model.npz`, `quality/build_features.npz` and (optional) `quality/locator.npz` in your Drive folder, plus `MODEL_API_KEY` in Colab Secrets for Muse Spark.
The last cell prints a link that opens the page in a new tab.

In [ ]:
import os, sys, glob, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
MODEL = f'{D}/quality/coach_model.npz'
assert os.path.exists(MODEL), 'coach_model.npz is missing from your Drive quality folder'
APP = '/content/retake_coach'
os.makedirs(f'{APP}/static', exist_ok=True)
subprocess.run('pip install -q fastapi uvicorn python-multipart openai', shell=True, check=True)

In [ ]:
%%writefile /content/retake_coach/coach_core.py
"""Retake coach: is this fundus photo good enough to grade, and if not, what should the photographer fix?

It judges photo quality only and makes no diagnosis.

- Detection: Meta's DINOv2 (facebook/dinov2-base, frozen) runs locally. Four small logistic heads read its
  features plus three simple image checks: not good enough, focus, aim, obstruction. The heads are trained on
  DeepDRiD's photo-quality labels (CC BY-SA 4.0, https://github.com/deepdrdoc/DeepDRiD).
- Guard: an energy on the same features (a Gaussian model of ordinary fundus photos, i.e. Mahalanobis distance)
  flags inputs that do not look like a regular fundus photo, such as a widefield image or a random picture.
- Wording: Muse Spark (Meta Model API) writes the instruction for the problems the heads found. A hard cap on
  requests and dollars applies; past the cap, or on any error, fixed templates are used.
"""
import base64
import io
import json

import numpy as np
from PIL import Image
from scipy import ndimage

S = 448
PROBLEMS = ("focus", "aim", "obstruction")
DESCRIBE = {
    "focus": "out of focus or hazy: the small vessels are not sharp",
    "aim": "badly aimed: the optic disc and the macula are not both near the centre of the field",
    "obstruction": "partly covered: eyelid or eyelash shadow, glare, reflections or dust over the retina",
}
TEMPLATES = {
    "focus": "Refocus until the small vessels look sharp, then retake.",
    "aim": "Re-aim so the optic disc and the macula are both near the centre, then retake.",
    "obstruction": "Ask the patient to open wide (lift the eyelid if needed), wipe the lens, then retake.",
}
GENERIC = "Not good enough to grade: check focus, aim and eyelids, then retake."
MINOR = {"focus": "Focus could be sharper", "aim": "The aim is a little off centre", "obstruction": "Part of the view is shaded"}


def keep_text(minor, aim_text=None):
    if not minor:
        return "Good enough to grade. Move on to the next photo."
    where = aim_text.split(". ")[0] if (minor[0] == "aim" and aim_text and aim_text != TEMPLATES["aim"]) else MINOR[minor[0]]
    return f"Good enough to grade. {where}, but this photo is usable."
NOT_FUNDUS = "This does not look like a regular fundus photo. Check the camera mode or the file."
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])


# ---------- image preparation (identical to the pipeline the heads were trained on) ----------

def _open(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if isinstance(src, np.ndarray):
        return Image.fromarray(src).convert("RGB")
    return Image.open(src).convert("RGB")


def load_square(src, geom=False):
    """Crop to the retina, pad to a square, resize to 448 px. Returns float32 in [0, 1]. src: path, PIL image or uint8 array.
    geom=True also returns how original pixel coordinates map into the square (see to_square)."""
    img = _open(src)
    W0, H0 = img.size
    img.thumbnail((1024, 1024))
    a = np.asarray(img)
    ys, xs = np.nonzero(a.max(-1) > 15)
    y0 = x0 = 0
    if len(ys):                                   # an all-black frame keeps its full size
        y0, x0 = ys.min(), xs.min()
        a = a[y0:ys.max() + 1, x0:xs.max() + 1]
    h, w = a.shape[:2]
    side = max(h, w)
    sq = np.zeros((side, side, 3), np.uint8)
    sq[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = a
    x = np.asarray(Image.fromarray(sq).resize((S, S), Image.BILINEAR)).astype(np.float32) / 255
    if not geom:
        return x
    return x, dict(sx=img.size[0] / W0, sy=img.size[1] / H0, x0=int(x0), y0=int(y0),
                   px=(side - w) // 2, py=(side - h) // 2, side=side)


def to_square(X, Y, g):
    """Original pixel (X, Y) -> position in the square photo, as fractions (0..1) of its width and height."""
    return (X * g["sx"] - g["x0"] + g["px"]) / g["side"], (Y * g["sy"] - g["y0"] + g["py"]) / g["side"]


def retina_mask(x):
    return x.max(-1) > 15 / 255


def simple_checks(x):
    """Sharpness (Laplacian variance), brightness and contrast of the green channel inside the retina."""
    g, m = x[..., 1], ndimage.binary_erosion(retina_mask(x), iterations=8)
    if m.sum() < 100:                             # nearly black frame: measure the whole picture
        m = np.ones_like(m)
    return [ndimage.laplace(g)[m].var(), g[m].mean(), g[m].std()]


class Encoder:
    """Frozen DINOv2: class token + mean patch token (1,536 numbers per photo)."""

    def __init__(self, name="facebook/dinov2-base", device=None):
        import torch
        from transformers import AutoModel
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModel.from_pretrained(name).to(self.device, self.dtype).eval()

    def __call__(self, xs, bs=32, patches=False):
        """patches=True also returns the patch tokens as a (n, 32, 32, 768) float16 grid, used to locate the disc and macula."""
        out, grids = [], []
        with self.torch.no_grad():
            for i in range(0, len(xs), bs):
                b = self.torch.tensor((np.stack(xs[i:i + bs]) - MEAN) / STD).permute(0, 3, 1, 2)
                h = self.model(pixel_values=b.to(self.device, self.dtype)).last_hidden_state
                out.append(self.torch.cat([h[:, 0], h[:, 1:].mean(1)], 1).float().cpu())
                if patches:
                    n = int(round((h.shape[1] - 1) ** 0.5))
                    grids.append(h[:, 1:].reshape(len(b), n, n, -1).half().cpu())
        X = self.torch.cat(out).numpy()
        return (X, self.torch.cat(grids).numpy()) if patches else X


def describe(srcs, encoder, bs=32, workers=8):
    """Photos -> (DINOv2 features, simple checks). Loads in parallel, encodes in batches."""
    from concurrent.futures import ThreadPoolExecutor
    X, C = [], []
    for i in range(0, len(srcs), bs):
        with ThreadPoolExecutor(workers) as ex:
            xs = list(ex.map(load_square, srcs[i:i + bs]))
        X.append(encoder(xs, bs))
        C += [simple_checks(x) for x in xs]
    return np.concatenate(X), np.array(C)


# ---------- detector heads and energy guard ----------

def _sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class Coach:
    HEADS = ("retake",) + PROBLEMS
    RULES = ("overall", "problems", "both")   # what triggers a retake: the overall head, any problem head, or both

    def __init__(self, mu, sd, W, b, thr, g_mean=None, g_white=None, g_thr=None, rule="overall", locator=None):
        assert rule in self.RULES
        self.mu, self.sd, self.W, self.b, self.thr, self.rule = mu, sd, W, b, np.asarray(thr, float), rule
        self.g_mean, self.g_white, self.g_thr = g_mean, g_white, g_thr
        self.locator = locator                   # optional Locator; saved in its own file

    @classmethod
    def fit(cls, X, checks, labels, C=0.05):
        """labels: dict head -> bool array; C: one value or a dict head -> value (smaller = stronger regularization).
        Thresholds start at 0.5; set coach.thr from validation photos."""
        from sklearn.linear_model import LogisticRegression
        mu, sd = checks.mean(0), checks.std(0) + 1e-12
        F = np.c_[X, (checks - mu) / sd]
        W, b = [], []
        for h in cls.HEADS:
            clf = LogisticRegression(C=C[h] if isinstance(C, dict) else C, max_iter=5000).fit(F, labels[h])
            W.append(clf.coef_[0])
            b.append(clf.intercept_[0])
        return cls(mu, sd, np.array(W), np.array(b), np.full(len(cls.HEADS), 0.5))

    def probs(self, X, checks):
        """(n, 4) probabilities in HEADS order."""
        return _sigmoid(np.c_[X, (checks - self.mu) / self.sd] @ self.W.T + self.b)

    def set_guard(self, X_ref, X_cal, quantile=0.99):
        """Energy guard fitted on ordinary fundus photos (X_ref); threshold = the 99th percentile on other ones (X_cal)."""
        from sklearn.covariance import LedoitWolf
        lw = LedoitWolf().fit(np.asarray(X_ref, np.float64))
        lam, V = np.linalg.eigh(lw.covariance_)               # whiten from the covariance itself:
        lam = np.maximum(lam, lam.max() * 1e-6)               # tiny directions are floored, never inverted to inf
        self.g_mean, self.g_white = lw.location_, V / np.sqrt(lam)
        self.g_thr = float(np.quantile(self.energy(X_cal), quantile))
        return self

    def energy(self, X):
        """Mahalanobis energy per feature dimension: about 1 for typical fundus photos, large for unfamiliar inputs."""
        return (((X - self.g_mean) @ self.g_white) ** 2).sum(1) / X.shape[1]

    def verdict(self, X, checks, patches=None, max_problems=2):
        """patches (from Encoder(..., patches=True)) adds measured disc and macula positions when a locator is set."""
        P = self.probs(X, checks)
        E = self.energy(X) if self.g_mean is not None else np.zeros(len(X))
        L = self.locator.locate(patches) if (self.locator is not None and patches is not None) else [None] * len(X)
        out = []
        for p, e, loc in zip(P, E, L):
            fundus = self.g_thr is None or bool(e <= self.g_thr)
            margin = _logit(p[1:]) - _logit(self.thr[1:])          # how far each problem head is past its threshold
            overall, any_problem = bool(p[0] >= self.thr[0]), bool((margin > 0).any())
            retake = {"overall": overall, "problems": any_problem, "both": overall and any_problem}[self.rule]
            order = [int(i) for i in np.argsort(-margin)]
            problems = [PROBLEMS[i] for i in order if margin[i] > 0][:max_problems] if retake else []
            if retake and not problems:                              # always give one concrete thing to fix
                problems = [PROBLEMS[order[0]]]
            v = dict(looks_like_fundus=fundus, retake=retake and fundus, problems=problems if fundus else [], minor=[],
                     p_not_good=float(p[0]), scores={k: float(v) for k, v in zip(PROBLEMS, p[1:])}, energy=float(e))
            if loc is not None and fundus:
                dd = self.locator.dd
                # a kept photo gets an aim note only when the aim head and the measured positions both say off centre
                both_found = loc["disc"] is not None and loc["macula"] is not None
                if not retake and margin[1] > 0 and both_found and aim_instruction(loc, dd) != TEMPLATES["aim"]:
                    v["minor"] = ["aim"]
                v.update(disc=loc["disc"], macula=loc["macula"], dd=dd, positions=positions_text(loc, dd),
                         aim_text=aim_instruction(loc, dd) if "aim" in v["problems"] + v["minor"] else None)
            out.append(v)
        return out

    def save(self, path):
        g = {} if self.g_mean is None else dict(g_mean=self.g_mean, g_white=self.g_white.astype(np.float32), g_thr=self.g_thr)
        np.savez(path, mu=self.mu, sd=self.sd, W=self.W, b=self.b, thr=self.thr, heads=np.array(self.HEADS),
                 rule=np.array(self.rule), **g)

    @classmethod
    def load(cls, path):
        z = np.load(path)
        assert tuple(z["heads"]) == cls.HEADS
        g = dict(g_mean=z["g_mean"], g_white=z["g_white"].astype(np.float64), g_thr=float(z["g_thr"])) if "g_mean" in z else {}
        return cls(z["mu"], z["sd"], z["W"], z["b"], z["thr"], rule=str(z["rule"]) if "rule" in z else "overall", **g)


# ---------- where are the optic disc and the macula? ----------

class Locator:
    """A linear map from DINOv2 patch tokens to two heatmaps (optic disc, fovea), fitted by ridge regression on
    IDRiD's expert-marked centres. Positions are fractions (0..1) of the square photo; dd is one disc diameter
    in the same units, so distances can be read the way DeepDRiD's field-definition score reads them."""
    NAMES = ("disc", "macula")

    def __init__(self, W, peak_min, dd):
        self.W, self.peak_min, self.dd = np.asarray(W, np.float32), np.asarray(peak_min, float), float(dd)

    @staticmethod
    def design(T):
        """(n, g, g, d) tokens -> (n, g*g, d+1) with a constant column for the bias."""
        T = np.asarray(T, np.float32).reshape(T.shape[0], -1, T.shape[-1])
        return np.concatenate([T, np.ones(T.shape[:2] + (1,), np.float32)], -1)

    def heatmaps(self, T, bs=32):
        g = T.shape[1]
        return np.concatenate([(self.design(T[i:i + bs]) @ self.W).reshape(-1, g, g, len(self.NAMES))
                               for i in range(0, len(T), bs)])

    def locate(self, T):
        out = []
        for hm in self.heatmaps(T):
            g, found = hm.shape[0], {}
            for k, name in enumerate(self.NAMES):
                m = ndimage.uniform_filter(hm[..., k], 3)
                i, j = np.unravel_index(np.argmax(m), m.shape)
                i0, i1, j0, j1 = max(i - 1, 0), min(i + 2, g), max(j - 1, 0), min(j + 2, g)
                win = np.clip(hm[i0:i1, j0:j1, k], 0, None) + 1e-9          # sub-patch refinement
                ii, jj = np.mgrid[i0:i1, j0:j1]
                u, v = ((jj * win).sum() / win.sum() + 0.5) / g, ((ii * win).sum() / win.sum() + 0.5) / g
                found[name] = (float(u), float(v)) if m[i, j] >= self.peak_min[k] else None
                found[name + "_peak"] = float(m[i, j])
            out.append(found)
        return out

    def save(self, path):
        np.savez(path, W=self.W, peak_min=self.peak_min, dd=self.dd)

    @classmethod
    def load(cls, path):
        z = np.load(path)
        return cls(z["W"], z["peak_min"], float(z["dd"]))


SIDES = ["right", "lower right", "bottom", "lower left", "left", "upper left", "top", "upper right"]


def side_of(u, v):
    """Which edge of the picture a point is toward, in the words a technician sees on screen."""
    ang = np.degrees(np.arctan2(v - 0.5, u - 0.5)) % 360          # 0 = right, 90 = bottom (image y grows downward)
    return SIDES[int(((ang + 22.5) % 360) // 45)]


def positions_text(loc, dd):
    """Measured positions as plain facts, e.g. 'the optic disc is near the centre; the macula is near the right edge'."""
    parts = []
    for name, label in [("disc", "the optic disc"), ("macula", "the macula")]:
        p = loc.get(name)
        if p is None:
            parts.append(f"{label} is not in the picture")
        elif np.hypot(p[0] - 0.5, p[1] - 0.5) <= 2 * dd:
            parts.append(f"{label} is near the centre")
        else:
            parts.append(f"{label} is near the {side_of(*p)} edge")
    return "; ".join(parts)


def aim_instruction(loc, dd):
    """A direction measured from the photo, never guessed."""
    off = {n: loc[n] for n in ("disc", "macula") if loc.get(n) and np.hypot(loc[n][0] - 0.5, loc[n][1] - 0.5) > 2 * dd}
    missing = [n for n in ("disc", "macula") if loc.get(n) is None]
    name = {"disc": "optic disc", "macula": "macula"}
    if missing:
        return f"The {name[missing[0]]} is not in the picture. Re-aim so the optic disc and the macula are both near the centre, then retake."
    if len(off) == 1:
        (n, p), = off.items()
        return f"The {name[n]} is near the {side_of(*p)} edge of the picture. Re-aim so it moves toward the centre, then retake."
    if len(off) == 2:
        mid = ((off["disc"][0] + off["macula"][0]) / 2, (off["disc"][1] + off["macula"][1]) / 2)
        return f"The disc and macula both sit toward the {side_of(*mid)} of the picture. Re-aim so they move toward the centre, then retake."
    return TEMPLATES["aim"]


# ---------- wording: Muse Spark, capped, with templates as the fallback ----------

PROMPT = """You coach a technician who takes retinal (fundus) photos for eye screening. An automatic photo-quality
check found that this photo needs a retake because it is:
{found}
{facts}Write ONE instruction telling the technician what to do: plain words, starting with a verb, at most 20 words,
fixing only the problems listed. It is not something to say to the patient. Say nothing about anything else, including framing or aim if aim is not listed. Look at the
photo only to say where a shadow, glare or haze is. Never guess where the optic disc or the macula is or which way to
aim: use only the measured positions given above. If one of them is not in the picture, say to re-aim so it comes into
view, without a direction. Do not mention disease or diagnosis.
Reply with JSON only: {{"instruction": "..."}}"""


def template(problems, aim_text=None):
    if not problems:
        return GENERIC
    parts = [aim_text if (p == "aim" and aim_text) else TEMPLATES[p] for p in problems]
    return " ".join([t.replace(", then retake.", ".") for t in parts[:-1]] + parts[-1:])   # say "then retake" once


def _data_url(src, px):
    img = _open(src)
    img.thumbnail((px, px))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _json_field(text, key):
    try:
        return str(json.loads(text[text.index("{"):text.rindex("}") + 1])[key]).strip() or None
    except (ValueError, KeyError, TypeError):
        return None


class Wording:
    PRICE_IN, PRICE_OUT = 1.25e-6, 4.25e-6       # muse-spark-1.3, Standard mode, $ per token

    def __init__(self, client=None, model="muse-spark-1.3", effort="minimal", max_requests=20, max_usd=0.10, px=768):
        self.client, self.model, self.effort, self.px = client, model, effort, px
        self.max_requests, self.max_usd = max_requests, max_usd
        self.n, self.spent, self.worst = 0, 0.0, 0.01  # assume a call may cost 1 cent until a dearer one is seen

    def can_call(self):
        return self.client is not None and self.n < self.max_requests and self.spent + self.worst <= self.max_usd

    def __call__(self, src, problems, positions=None, aim_text=None):
        """positions: measured facts such as 'the optic disc is near the centre; the macula is near the right edge'."""
        if not problems or not self.can_call():
            return dict(text=template(problems, aim_text), source="template", cost=0.0)
        found = "\n".join(f"- {DESCRIBE[p]}" for p in problems)
        positions = positions if "aim" in problems else None     # positions only matter, and only invite comment, for aim
        facts = f"Measured positions in the picture: {positions}.\n" if positions else ""
        self.n += 1
        try:
            resp = self.client.chat.completions.create(
                model=self.model, extra_body={"reasoning_effort": self.effort},
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": PROMPT.format(found=found, facts=facts)},
                    {"type": "image_url", "image_url": {"url": _data_url(src, self.px)}}]}])
        except Exception as e:  # network, quota, bad key: the technician still gets an instruction
            return dict(text=template(problems, aim_text), source=f"template (Muse Spark error: {type(e).__name__})", cost=0.0)
        cost = resp.usage.prompt_tokens * self.PRICE_IN + resp.usage.completion_tokens * self.PRICE_OUT
        self.spent += cost
        self.worst = max(self.worst, cost)
        text = _json_field(resp.choices[0].message.content or "", "instruction")
        return dict(text=text or template(problems, aim_text), source="muse-spark" if text else "template (unparsed reply)", cost=cost)


In [ ]:
%%writefile /content/retake_coach/app.py
"""Retake Coach demo server: one page, one endpoint.

    pip install -r requirements.txt
    export MODEL_API_KEY=...            # optional: without it, instructions come from fixed templates
    python app.py                       # uses models/coach_model.npz and models/locator.npz

The photo never leaves this computer for detection: DINOv2 and the heads run here. Only when a retake is needed,
and only within the Muse Spark budget below, the photo is sent to the Meta Model API to word the instruction.

Budget (environment variables, dollars at Muse Spark 1.3 Standard prices):
    MUSE_DAILY_USD      default 0.25    MUSE_DAILY_REQUESTS default 60    MUSE_TOTAL_USD default 1.00
Past any cap the page keeps working with template instructions.
"""
import argparse
import base64
import datetime
import io
import os
import threading
import time
from pathlib import Path

import numpy as np
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import FileResponse, JSONResponse
from PIL import Image, UnidentifiedImageError

import coach_core as cc

HERE = Path(__file__).parent
MAX_BYTES = 25 * 1024 * 1024
IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


class Budget:
    """Muse Spark spending: a fresh daily allowance, never past the total cap for this server's lifetime."""

    def __init__(self, client, daily_usd, daily_requests, total_usd):
        self.client, self.daily_usd, self.daily_requests, self.total_usd = client, daily_usd, daily_requests, total_usd
        self.spent_before_today, self.day, self.words = 0.0, None, None
        self.lock = threading.Lock()

    def today(self):
        with self.lock:
            d = datetime.date.today()
            if d != self.day:
                if self.words is not None:
                    self.spent_before_today += self.words.spent
                left = max(0.0, self.total_usd - self.spent_before_today)
                self.day = d
                self.words = cc.Wording(self.client, max_requests=self.daily_requests, max_usd=min(self.daily_usd, left))
            return self.words

    def status(self):
        w = self.today()
        return dict(muse=self.client is not None, can_call=w.can_call(), spent_today=round(w.spent, 4),
                    cap_today=round(w.max_usd, 4), requests_today=w.n, request_cap=w.max_requests,
                    spent_total=round(self.spent_before_today + w.spent, 4), cap_total=self.total_usd)


def preview(x):
    """The 448 px square the detector saw, as a small JPEG data URL for the page."""
    buf = io.BytesIO()
    Image.fromarray((x * 255).astype(np.uint8)).save(buf, "JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def create_app(coach, encoder, budget, examples_dir=None):
    app = FastAPI(title="Retake Coach")
    lock, wlock = threading.Lock(), threading.Lock()   # one photo at a time through the encoder; exact budget counts
    examples = Path(examples_dir) if examples_dir else None

    @app.get("/")
    def index():
        return FileResponse(HERE / "static" / "index.html")

    @app.get("/api/status")
    def status():
        return dict(budget=budget.status(), thresholds=dict(zip(cc.Coach.HEADS, map(float, coach.thr))),
                    energy_limit=coach.g_thr)

    @app.get("/api/examples")
    def list_examples():
        """Sample photos, with an optional note per file from examples/labels.json (e.g. the dataset's own label)."""
        if not examples or not examples.is_dir():
            return []
        notes = {}
        if (examples / "labels.json").is_file():
            import json
            notes = json.loads((examples / "labels.json").read_text())
        return [dict(name=p.name, note=notes.get(p.name, "")) for p in sorted(examples.iterdir()) if p.suffix.lower() in IMAGE_TYPES]

    @app.get("/examples/{name}")
    def example(name: str):
        p = (examples / name).resolve() if examples else None
        if not p or p.parent != examples.resolve() or not p.is_file():
            raise HTTPException(404, "No such sample photo.")
        return FileResponse(p)

    @app.post("/api/check")
    def check(photo: UploadFile = File(...), use_muse: bool = Form(True)):
        data = photo.file.read(MAX_BYTES + 1)
        if len(data) > MAX_BYTES:
            raise HTTPException(413, "That file is over 25 MB. Export the photo as a JPEG and try again.")
        try:
            img = Image.open(io.BytesIO(data))
            img.load()
            img = img.convert("RGB")
        except (UnidentifiedImageError, OSError, Image.DecompressionBombError):
            raise HTTPException(400, "That file is not a photo the check can read. Use a JPEG or PNG.")

        t0 = time.perf_counter()
        x = cc.load_square(img)
        with lock:
            X, grid = encoder([x], patches=True) if coach.locator is not None else (encoder([x]), None)
        v = coach.verdict(X, np.array([cc.simple_checks(x)]), patches=grid)[0]
        t1 = time.perf_counter()

        if not v["looks_like_fundus"]:
            verdict, words = "not_fundus", dict(text=cc.NOT_FUNDUS, source="template", cost=0.0)
        elif v["retake"]:
            verdict = "retake"
            if use_muse:
                with wlock:
                    words = budget.today()(img, v["problems"], positions=v.get("positions"), aim_text=v.get("aim_text"))
            else:
                words = dict(text=cc.template(v["problems"], v.get("aim_text")), source="template", cost=0.0)
        else:
            verdict, words = "keep", dict(text=cc.keep_text(v["minor"], v.get("aim_text")), source="template", cost=0.0)
        t2 = time.perf_counter()

        checks = {k: dict(score=round(v["scores"][k], 3), threshold=round(float(t), 3), found=k in v["problems"],
                          minor=k in v["minor"])
                  for k, t in zip(cc.PROBLEMS, coach.thr[1:])}
        return JSONResponse(dict(
            verdict=verdict, instruction=words["text"], source=words["source"], cost=round(words["cost"], 5),
            problems=v["problems"], checks=checks, p_not_good=round(v["p_not_good"], 3),
            retake_threshold=round(float(coach.thr[0]), 3), rule=coach.rule, energy=round(v["energy"], 2),
            energy_limit=round(float(coach.g_thr), 2) if coach.g_thr is not None else None,
            disc=v.get("disc"), macula=v.get("macula"), dd=v.get("dd"), positions=v.get("positions"),
            ms=dict(detect=round(1000 * (t1 - t0)), wording=round(1000 * (t2 - t1))),
            preview=preview(x), budget=budget.status()))

    return app


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="models/coach_model.npz")
    ap.add_argument("--locator", default="models/locator.npz", help="optional: disc and macula positions for aim directions")
    ap.add_argument("--examples", default="examples")
    ap.add_argument("--host", default="127.0.0.1")
    ap.add_argument("--port", type=int, default=8000)
    a = ap.parse_args()

    client = None
    if os.environ.get("MODEL_API_KEY"):
        from openai import OpenAI
        client = OpenAI(base_url="https://api.meta.ai/v1", api_key=os.environ["MODEL_API_KEY"], timeout=30)
    budget = Budget(client, float(os.environ.get("MUSE_DAILY_USD", 0.25)), int(os.environ.get("MUSE_DAILY_REQUESTS", 60)),
                    float(os.environ.get("MUSE_TOTAL_USD", 1.00)))
    print("loading DINOv2 ...")
    coach = cc.Coach.load(a.model)
    if os.path.exists(a.locator):
        coach.locator = cc.Locator.load(a.locator)
    app = create_app(coach, cc.Encoder(), budget, a.examples)
    import uvicorn
    uvicorn.run(app, host=a.host, port=a.port)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/retake_coach/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Retake Coach</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible+Next:wght@400;600;800&display=swap" rel="stylesheet">
<style>
/* Dark on purpose: fundus cameras are used in dimmed rooms, so the screen must not light the room up. */
:root {
  --room: #171c22;      /* exam-room slate */
  --panel: #202830;
  --line: #34414d;
  --ink: #e6ebf0;
  --ink-2: #9eacba;
  --keep: #63c29e;      /* the camera's green fixation light */
  --retake: #f4b740;    /* amber "not ready" lamp */
  --away: #aab5c2;
  --focus: #8fc1ff;
  color-scheme: dark;
}
* { box-sizing: border-box; }
html, body { margin: 0; background: var(--room); color: var(--ink); }
body {
  font-family: "Atkinson Hyperlegible Next", "Atkinson Hyperlegible", system-ui, sans-serif;
  font-size: 17px; line-height: 1.5;
  -webkit-font-smoothing: antialiased;
}
:focus-visible { outline: 3px solid var(--focus); outline-offset: 3px; border-radius: 6px; }
.sr { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }

header {
  display: flex; align-items: center; justify-content: space-between; gap: 16px; flex-wrap: wrap;
  padding: 18px clamp(16px, 4vw, 40px); border-bottom: 1px solid var(--line);
}
.mark { display: flex; align-items: center; gap: 10px; font-weight: 800; font-size: 21px; letter-spacing: -0.01em; }
.mark svg { width: 26px; height: 26px; }
.status { display: flex; gap: 8px; flex-wrap: wrap; font-size: 14px; color: var(--ink-2); }
.status span { border: 1px solid var(--line); border-radius: 999px; padding: 4px 12px; }
.status b { color: var(--ink); font-weight: 600; }

main {
  display: grid; grid-template-columns: minmax(280px, 540px) minmax(0, 560px); column-gap: clamp(24px, 5vw, 72px);
  row-gap: 24px; grid-template-areas: "photo result" "samples result" "session session";
  justify-content: center; align-items: start; padding: clamp(20px, 4vw, 48px) clamp(16px, 4vw, 40px);
}
.photo { grid-area: photo; } .result { grid-area: result; } .samples { grid-area: samples; } .session { grid-area: session; }
@media (max-width: 900px) {
  main { grid-template-columns: minmax(0, 520px); grid-template-areas: "photo" "result" "samples" "session"; }
}

/* ---- viewfinder: the photo in its circle, ringed by the three checks ---- */
.viewfinder { position: relative; aspect-ratio: 660 / 520; width: 100%; }
.viewfinder svg { width: 100%; height: 100%; display: block; overflow: visible; }
.arc { fill: none; stroke: var(--line); stroke-width: 4; stroke-linecap: round; }
.arc.clear { stroke: var(--keep); }
.arc.found { stroke: var(--retake); stroke-width: 12; }
.arc.minor { stroke: var(--retake); stroke-width: 4; }
.arc.off { stroke: var(--away); stroke-dasharray: 3 9; }
.arc-label { fill: var(--ink-2); font-size: 26px; font-weight: 600; }
.arc-label.found { fill: var(--retake); }
.rim { fill: none; stroke: var(--line); stroke-width: 2; stroke-dasharray: 6 8; }
.has-photo .rim { stroke-dasharray: none; stroke: #0000; }
.pos-disc { fill: none; stroke: var(--ink); stroke-width: 3; }
.pos-macula { fill: var(--ink); stroke: var(--room); stroke-width: 2; }
.pos-zone { fill: none; stroke: var(--ink); stroke-width: 1.5; stroke-dasharray: 5 7; opacity: 0.8; }
.pos-label { fill: var(--ink); font-size: 18px; font-weight: 600; paint-order: stroke; stroke: var(--room); stroke-width: 4px; }
.scan { fill: none; stroke: var(--focus); stroke-width: 4; stroke-linecap: round; opacity: 0; transform-box: fill-box; transform-origin: center; }
.checking .scan { opacity: 1; animation: spin 1.1s linear infinite; }
@keyframes spin { to { transform: rotate(360deg); } }
.drawn .arc { animation: draw 0.7s ease-out both; }
@keyframes draw { from { stroke-dashoffset: var(--len); } to { stroke-dashoffset: 0; } }
@media (prefers-reduced-motion: reduce) {
  .checking .scan { animation: none; stroke-dasharray: 40 1000; }
  .drawn .arc { animation: none; }
}

.drop {
  position: absolute; inset: 11.5% 19.7%; border-radius: 50%; display: grid; place-content: center; gap: 14px;
  text-align: center; cursor: pointer; padding: 12%;
}
.drop .lead { font-size: 21px; font-weight: 600; }
.drop .btn {
  justify-self: center; background: var(--ink); color: var(--room); font-weight: 800; border-radius: 999px; padding: 10px 22px;
}
.has-photo .drop { opacity: 0; background: rgba(23, 28, 34, 0.78); transition: opacity 0.15s; }
.has-photo .drop:hover, .has-photo .drop:focus-within, .dragging .drop { opacity: 1; }
.dragging .rim { stroke: var(--focus); stroke-dasharray: none; }
.error { color: var(--retake); min-height: 1.5em; margin: 12px 0 0; text-align: center; }

.samples h2, .session h2, .result h3 { font-size: 15px; font-weight: 600; color: var(--ink-2); margin: 0 0 10px; }
.thumbs { display: flex; gap: 10px; flex-wrap: wrap; }
.thumbs button {
  width: 56px; height: 56px; padding: 0; border-radius: 50%; overflow: hidden; border: 2px solid var(--line);
  background: #000; cursor: pointer;
}
.thumbs button:hover { border-color: var(--ink-2); }
.thumbs img { width: 100%; height: 100%; object-fit: cover; display: block; }
.thumbs .keep { border-color: var(--keep); }
.thumbs .retake { border-color: var(--retake); }
.thumbs .not_fundus { border-color: var(--away); border-style: dashed; }

/* ---- result ---- */
.result { min-height: 200px; }
.idle h1 { font-size: 33px; line-height: 1.2; margin: 8px 0 12px; font-weight: 800; letter-spacing: -0.015em; }
.idle p { color: var(--ink-2); max-width: 46ch; margin: 0 0 12px; }
.verdict { display: flex; align-items: center; gap: 10px; font-size: 26px; font-weight: 800; margin: 4px 0 6px; }
.verdict svg { width: 26px; height: 26px; flex: none; }
.v-keep .verdict { color: var(--keep); }
.v-retake .verdict { color: var(--retake); }
.v-not_fundus .verdict { color: var(--away); }
.instruction { font-size: clamp(26px, 3.2vw, 34px); line-height: 1.22; font-weight: 800; letter-spacing: -0.012em; margin: 0 0 10px; max-width: 24ch; }
.source { color: var(--ink-2); font-size: 14px; margin: 0 0 28px; }
.rows { margin: 0; display: grid; grid-template-columns: max-content 1fr; gap: 10px 18px; align-items: baseline; }
.rows dt { color: var(--ink-2); }
.rows dd { margin: 0; }
.rows .found { color: var(--retake); font-weight: 600; }
.meter { position: relative; height: 6px; border-radius: 3px; background: var(--line); margin-top: 6px; max-width: 260px; }
.meter i { position: absolute; inset: 0 auto 0 0; border-radius: 3px; background: var(--ink-2); }
.meter.found i { background: var(--retake); }
.meter b { position: absolute; top: -4px; width: 2px; height: 14px; background: var(--ink); }
.small { font-size: 14px; color: var(--ink-2); }
.toggle { display: flex; gap: 10px; align-items: flex-start; margin-top: 26px; font-size: 15px; color: var(--ink-2); max-width: 52ch; }
.toggle input { width: 18px; height: 18px; margin-top: 3px; accent-color: var(--keep); }

.session { border-top: 1px solid var(--line); padding-top: 18px; }
.session .count { color: var(--ink-2); font-size: 15px; margin: -4px 0 12px; }
footer { padding: 8px clamp(16px, 4vw, 40px) 32px; color: var(--ink-2); font-size: 14px; max-width: 1150px; margin: 0 auto; }
footer p { margin: 4px 0; }
</style>
</head>
<body>
<header>
  <div class="mark">
    <svg viewBox="0 0 26 26" aria-hidden="true"><circle cx="13" cy="13" r="11" fill="none" stroke="#63c29e" stroke-width="2.4"/>
      <circle cx="13" cy="13" r="4" fill="#f4b740"/></svg>
    Retake Coach
  </div>
  <div class="status" aria-live="polite">
    <span>Detector: <b>DINOv2 on this computer</b></span>
    <span id="wording-status">Instructions: <b>checking…</b></span>
  </div>
</header>

<main>
  <section class="photo" aria-label="Photo">
    <div class="viewfinder" id="vf">
      <svg viewBox="-80 -10 660 520" aria-hidden="true">
        <defs><clipPath id="disc"><circle cx="250" cy="250" r="200"/></clipPath></defs>
        <circle class="rim" cx="250" cy="250" r="200"/>
        <image id="photo" x="50" y="50" width="400" height="400" clip-path="url(#disc)" preserveAspectRatio="xMidYMid slice"/>
        <g id="arcs"></g>
        <g id="positions"></g>
        <circle class="scan" cx="250" cy="250" r="222" stroke-dasharray="90 1300"/>
      </svg>
      <label class="drop" for="file">
        <span class="lead" id="drop-lead">Drop the photo you just took</span>
        <span class="btn">Choose a photo</span>
      </label>
      <input class="sr" type="file" id="file" accept="image/*">
    </div>
    <p class="error" id="error" role="alert"></p>
  </section>
  <section class="samples" id="samples" hidden>
    <h2>Sample photos</h2>
    <div class="thumbs" id="sample-thumbs"></div>
  </section>

  <section class="result idle" id="result" aria-live="polite">
    <h1>Check each fundus photo before the patient leaves the chair.</h1>
    <p>Retake Coach says whether the photo is good enough to grade. If it isn't, it tells you what to fix: focus, aim, or something blocking the view.</p>
    <p>It checks photo quality only. It does not look for disease.</p>
  </section>

  <section class="session" id="session" hidden>
    <h2>This session</h2>
    <p class="count" id="session-count"></p>
    <div class="thumbs" id="session-thumbs"></div>
  </section>
</main>

<footer>
  <p>Detector: Meta's DINOv2 image features with small heads trained on DeepDRiD photo-quality labels (CC BY-SA 4.0). It runs on this computer.</p>
  <p>Instructions: Muse Spark through the Meta Model API, under a daily spending cap. Without it, standard instructions are used.</p>
</footer>

<script>
const PROBLEMS = { focus: [-56, 56], aim: [64, 176], obstruction: [184, 296] };
const WORDS = {
  focus: ['Blurry', 'Sharp'], aim: ['Off centre', 'Centred'], obstruction: ['Shadow or glare', 'Clear view']
};
const NAMES = { focus: 'Focus', aim: 'Aim', obstruction: 'Obstruction' };
const ICONS = {
  keep: '<svg viewBox="0 0 26 26"><circle cx="13" cy="13" r="11" fill="none" stroke="currentColor" stroke-width="2.6"/><path d="M8 13.5l3.4 3.4L18.5 9" fill="none" stroke="currentColor" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/></svg>',
  retake: '<svg viewBox="0 0 26 26"><path d="M13 2.5l11 19.5H2z" fill="none" stroke="currentColor" stroke-width="2.4" stroke-linejoin="round"/><path d="M13 10v5.5" stroke="currentColor" stroke-width="2.6" stroke-linecap="round"/><circle cx="13" cy="18.8" r="1.5" fill="currentColor"/></svg>',
  not_fundus: '<svg viewBox="0 0 26 26"><circle cx="13" cy="13" r="11" fill="none" stroke="currentColor" stroke-width="2.4" stroke-dasharray="4 3"/><path d="M9 9l8 8M17 9l-8 8" stroke="currentColor" stroke-width="2.4" stroke-linecap="round"/></svg>'
};
const HEAD = { keep: 'Keep', retake: 'Retake', not_fundus: 'Not a fundus photo' };
const $ = id => document.getElementById(id);
const vf = $('vf'), result = $('result');
let session = [], useMuse = true, museAvailable = false;

function polar(r, deg) { const a = (deg - 90) * Math.PI / 180; return [250 + r * Math.cos(a), 250 + r * Math.sin(a)]; }
function arcPath(r, a0, a1) {
  const [x0, y0] = polar(r, a0), [x1, y1] = polar(r, a1);
  return `M${x0.toFixed(1)} ${y0.toFixed(1)} A${r} ${r} 0 ${a1 - a0 > 180 ? 1 : 0} 1 ${x1.toFixed(1)} ${y1.toFixed(1)}`;
}
function drawArcs(states) {           // states: {focus: 'found'|'clear'|'off'|''}
  const g = $('arcs'); g.innerHTML = '';
  for (const [k, [a0, a1]] of Object.entries(PROBLEMS)) {
    const len = (Math.PI * 222 * (a1 - a0) / 180).toFixed(0);
    const p = document.createElementNS('http://www.w3.org/2000/svg', 'path');
    p.setAttribute('d', arcPath(222, a0, a1));
    p.setAttribute('class', 'arc ' + (states[k] || ''));
    p.style.setProperty('--len', len); p.style.strokeDasharray = states[k] === 'off' ? '' : `${len} ${len}`;
    g.appendChild(p);
    const mid = (a0 + a1) / 2, [x, y] = polar(250, mid);
    const t = document.createElementNS('http://www.w3.org/2000/svg', 'text');
    t.setAttribute('x', x.toFixed(1)); t.setAttribute('y', (y + 6).toFixed(1));
    t.setAttribute('text-anchor', Math.abs(Math.sin(mid * Math.PI / 180)) < 0.3 ? 'middle' : (x > 250 ? 'start' : 'end'));
    t.setAttribute('class', 'arc-label ' + (states[k] === 'found' ? 'found' : ''));
    t.textContent = NAMES[k];
    g.appendChild(t);
  }
}
drawArcs({});

function drawPositions(r) {          // measured optic disc (ring) and macula (dot); the 2-disc-diameter zone when aim is off
  const g = $('positions'); g.innerHTML = '';
  if (r.verdict === 'not_fundus' || !r.dd) return;
  const ns = 'http://www.w3.org/2000/svg', at = ([u, v]) => [50 + 400 * u, 50 + 400 * v];
  const add = (tag, attrs) => { const e = document.createElementNS(ns, tag); for (const k in attrs) e.setAttribute(k, attrs[k]); g.appendChild(e); return e; };
  if (r.checks.aim.found || r.checks.aim.minor) add('circle', { cx: 250, cy: 250, r: 800 * r.dd, class: 'pos-zone' });
  if (r.disc) { const [x, y] = at(r.disc); add('circle', { cx: x, cy: y, r: 200 * r.dd, class: 'pos-disc' }); }
  if (r.macula) { const [x, y] = at(r.macula); add('circle', { cx: x, cy: y, r: 7, class: 'pos-macula' }); }
}

function esc(s) { return String(s).replace(/[&<>"]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;' }[c])); }
function money(x) { return x === 0 ? '$0' : '$' + (x < 0.01 ? x.toFixed(4) : x.toFixed(2)); }

function setStatus(b) {
  museAvailable = b.muse;
  const el = $('wording-status');
  if (!b.muse) el.innerHTML = 'Instructions: <b>standard (no Muse Spark key)</b>';
  else if (!b.can_call) el.innerHTML = `Instructions: <b>standard (today's Muse Spark cap reached)</b>`;
  else el.innerHTML = `Instructions: <b>Muse Spark</b>, ${money(b.spent_today)} of ${money(b.cap_today)} used today`;
}

function render(r, note) {
  vf.classList.remove('checking'); vf.classList.add('has-photo');
  $('photo').setAttribute('href', r.preview);
  $('drop-lead').textContent = 'Drop or choose another photo';
  const states = {};
  for (const k of Object.keys(PROBLEMS)) states[k] = r.verdict === 'not_fundus' ? 'off' : (r.checks[k].found ? 'found' : r.checks[k].minor ? 'minor' : 'clear');
  vf.classList.remove('drawn'); void vf.offsetWidth; vf.classList.add('drawn');
  drawArcs(states);
  drawPositions(r);

  let rows = '';
  if (r.verdict !== 'not_fundus') {
    for (const k of Object.keys(PROBLEMS)) {
      const c = r.checks[k], found = c.found, flagged = c.found || c.minor;
      rows += `<dt>${NAMES[k]}</dt><dd><span class="${found ? 'found' : ''}">${WORDS[k][flagged ? 0 : 1]}${c.minor ? ' (still gradable)' : ''}</span>
        <span class="small">score ${c.score.toFixed(2)}, flagged from ${c.threshold.toFixed(2)}</span>
        <div class="meter ${found ? 'found' : ''}" aria-hidden="true"><i style="width:${(100 * c.score).toFixed(0)}%"></i><b style="left:${(100 * c.threshold).toFixed(0)}%"></b></div></dd>`;
    }
    rows += r.rule === 'problems'
      ? `<dt>Overall</dt><dd>Quality score ${r.p_not_good.toFixed(2)} <span class="small">for reference; a retake is asked only for a problem above</span></dd>`
      : `<dt>Overall</dt><dd>${r.p_not_good >= r.retake_threshold ? 'Not good enough to grade' : 'Good enough to grade'}
      <span class="small">score ${r.p_not_good.toFixed(2)}, retake from ${r.retake_threshold.toFixed(2)}</span></dd>`;
  }
  rows += `<dt>Fundus photo</dt><dd>${r.verdict === 'not_fundus' ? 'No, unlike the photos it knows' : 'Yes'}
    <span class="small">energy ${r.energy.toFixed(1)}, turned away above ${r.energy_limit.toFixed(1)}</span></dd>
    <dt>Time</dt><dd>Checked in ${r.ms.detect} ms${r.ms.wording > 50 ? `, instruction in ${(r.ms.wording / 1000).toFixed(1)} s` : ''}</dd>`;
  if (r.positions && r.verdict !== 'not_fundus' && (r.checks.aim.found || r.checks.aim.minor)) rows += `<dt>Positions</dt><dd>${esc(r.positions[0].toUpperCase() + r.positions.slice(1))}
    <span class="small">measured on the photo: ring = optic disc, dot = macula</span></dd>`;
  if (note) rows += `<dt>Sample</dt><dd>${esc(note.name)} ${note.text ? `<span class="small">${esc(note.text)}</span>` : ''}</dd>`;

  const src = r.source === 'muse-spark' ? `Written by Muse Spark for ${money(r.cost)}`
            : r.source.startsWith('template (') ? `Standard instruction: ${esc(r.source.slice(10, -1))}`
            : r.verdict === 'retake' ? 'Standard instruction' : '';
  result.className = 'result v-' + r.verdict;
  result.innerHTML = `
    <div class="verdict">${ICONS[r.verdict]}<span>${HEAD[r.verdict]}</span></div>
    <p class="instruction">${esc(r.instruction)}</p>
    <p class="source">${src}</p>
    <h3>What the check saw</h3>
    <dl class="rows">${rows}</dl>
    <label class="toggle"><input type="checkbox" id="muse" ${useMuse ? 'checked' : ''} ${museAvailable ? '' : 'disabled'}>
      <span>Word retake instructions with Muse Spark. This sends a retake photo to the Meta Model API. When it's off, standard instructions are used.</span></label>`;
  $('muse').addEventListener('change', e => { useMuse = e.target.checked; });
  setStatus(r.budget);
}

function addToSession(r, note) {
  session.unshift({ r, note });
  $('session').hidden = false;
  const n = session.length, re = session.filter(s => s.r.verdict === 'retake').length;
  const away = session.filter(s => s.r.verdict === 'not_fundus').length;
  $('session-count').textContent = `${n} checked, ${n - re - away} kept, ${re} to retake` + (away ? `, ${away} not ${away === 1 ? 'a fundus photo' : 'fundus photos'}` : '');
  const box = $('session-thumbs'); box.innerHTML = '';
  session.forEach((s, i) => {
    const b = document.createElement('button');
    b.className = s.r.verdict; b.title = HEAD[s.r.verdict];
    b.setAttribute('aria-label', `Photo ${n - i}: ${HEAD[s.r.verdict]}`);
    b.innerHTML = `<img alt="" src="${s.r.preview}">`;
    b.addEventListener('click', () => render(s.r, s.note));
    box.appendChild(b);
  });
}

async function check(blob, note) {
  $('error').textContent = '';
  vf.classList.add('checking');
  if (!vf.classList.contains('has-photo')) {
    vf.classList.add('has-photo'); $('photo').setAttribute('href', URL.createObjectURL(blob));
  }
  const fd = new FormData(); fd.append('photo', blob, blob.name || 'photo.jpg'); fd.append('use_muse', useMuse);   // the server decides whether Muse Spark is available
  try {
    const res = await fetch('/api/check', { method: 'POST', body: fd });
    const body = await res.json().catch(() => ({}));
    if (!res.ok) throw new Error(body.detail || 'The check did not finish. Try the photo again.');
    render(body, note); addToSession(body, note);
  } catch (e) {
    vf.classList.remove('checking');
    if (!session.length) { vf.classList.remove('has-photo'); $('photo').removeAttribute('href'); }
    $('error').textContent = e.message.includes('fetch') ? 'The check server is not reachable. Make sure app.py is running.' : e.message;
  }
}

$('file').addEventListener('change', e => { const f = e.target.files[0]; if (f) check(f, ''); e.target.value = ''; });
['dragenter', 'dragover'].forEach(t => vf.addEventListener(t, e => { e.preventDefault(); vf.classList.add('dragging'); }));
['dragleave', 'drop'].forEach(t => vf.addEventListener(t, e => { e.preventDefault(); vf.classList.remove('dragging'); }));
vf.addEventListener('drop', e => { const f = e.dataTransfer.files[0]; if (f) check(f, ''); });

fetch('/api/status').then(r => r.json()).then(s => setStatus(s.budget)).catch(() => {
  $('wording-status').innerHTML = 'Instructions: <b>server not reachable</b>';
});
fetch('/api/examples').then(r => r.json()).then(list => {
  if (!list.length) return;
  $('samples').hidden = false;
  for (const ex of list) {
    const b = document.createElement('button');
    b.setAttribute('aria-label', `Check sample photo ${ex.name}`); b.title = ex.name;
    b.innerHTML = `<img alt="" loading="lazy" src="/examples/${encodeURIComponent(ex.name)}">`;
    b.addEventListener('click', async () => {
      const blob = await (await fetch('/examples/' + encodeURIComponent(ex.name))).blob();
      blob.name = ex.name; check(blob, { name: ex.name.replace(/\.[^.]+$/, ''), text: ex.note });
    });
    $('sample-thumbs').appendChild(b);
  }
}).catch(() => {});
</script>
</body>
</html>


Sample photos for the page, saved to `demo_examples/` in your Drive so this runs once.
They are DeepDRiD test photos the tool judged the same way as the dataset's own labels, plus a widefield photo,
a fundus photo from the web and an everyday picture. DeepDRiD is CC BY-SA 4.0.

In [ ]:
EX = f'{D}/demo_examples'
EXTRA = ['354_r1', '354_l1']   # disc centred, macula at one edge: measured aim directions for the video
fresh = not os.path.exists(f'{EX}/labels.json')
missing = [i for i in EXTRA if not os.path.exists(f'{EX}/{i}.jpg')]
if fresh or missing:
    import numpy as np, pandas as pd, shutil
    from PIL import Image
    sys.path.insert(0, APP); import coach_core as cc
    DD = '/content/deepdrid_demo'
    T = 'regular_fundus_images/Online-Challenge1&2-Evaluation'
    UWF = 'ultra-widefield_images/ultra-widefield-validation/Images/34/34_l2.jpg'
    def sh(c): subprocess.run(c, shell=True, check=True)
    if not os.path.exists(DD):
        sh(f'git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/deepdrdoc/DeepDRiD.git {DD}')
        sh(f'cd {DD} && git config core.sparseCheckout true')
    def checkout(paths):
        with open(f'{DD}/.git/info/sparse-checkout', 'w') as f:
            f.write(''.join('/' + p + '\n' for p in paths))
        sh(f'cd {DD} && git read-tree -mu HEAD')
    checkout([f'{T}/Challenge2_labels.xlsx'])
    te = pd.read_excel(f'{DD}/{T}/Challenge2_labels.xlsx')
    G = np.load(f'{D}/quality/build_features.npz')
    coach = cc.Coach.load(MODEL)
    V = coach.verdict(G['te_X'], G['te_C'])
    bad = (te['Overall quality'] == 0).values
    lab = {'focus': (te['Clarity'] <= 6).values, 'aim': (te['Field definition'] <= 6).values, 'obstruction': (te['Artifact'] >= 6).values}
    def note(i):
        found = [k for k in cc.PROBLEMS if lab[k][i]]
        return 'DeepDRiD label: ' + ('not good enough' if bad[i] else 'good') + (f" ({', '.join(found)} flagged)" if found else '')
    picks = []
    if fresh:
        picks = [i for i, v in enumerate(V) if not v['retake'] and not bad[i] and not any(lab[k][i] for k in lab)][:2]
        for k in cc.PROBLEMS:
            picks += [i for i, v in enumerate(V) if v['retake'] and v['problems'][:1] == [k] and lab[k][i] and bad[i]][:2]
    picks += [int(np.where(te.image_id == i)[0][0]) for i in (EXTRA if fresh else missing) if (te.image_id == i).any()]
    picks = list(dict.fromkeys(picks))
    files = [f"{T}/Images/{te.image_id[i].split('_')[0]}/{te.image_id[i]}.jpg" for i in picks]
    checkout([f'{T}/Challenge2_labels.xlsx', UWF] + files)   # (UWF is small; fetched again harmlessly)
    os.makedirs(EX, exist_ok=True)
    notes = {} if fresh else json.load(open(f'{EX}/labels.json'))
    for i, f in zip(picks, files):
        shutil.copy(f'{DD}/{f}', f'{EX}/{te.image_id[i]}.jpg'); notes[f'{te.image_id[i]}.jpg'] = note(i)
    if fresh:
        shutil.copy(f'{DD}/{UWF}', f'{EX}/widefield_34_l2.jpg'); notes['widefield_34_l2.jpg'] = 'DeepDRiD widefield photo (a different kind of camera)'
        import skimage.data as sd
        for name, fn, txt in [('web_retina.jpg', 'retina', 'Fundus photo from the web (scikit-image sample)'),
                              ('everyday_cat.jpg', 'chelsea', 'Everyday picture (scikit-image sample)')]:
            try:
                Image.fromarray(np.asarray(getattr(sd, fn)())[..., :3]).save(f'{EX}/{name}', quality=92); notes[name] = txt
            except Exception as e:
                print('skipped', name, type(e).__name__)
    json.dump(notes, open(f'{EX}/labels.json', 'w'), indent=1)
subprocess.run(f'rm -rf {APP}/examples && cp -r "{EX}" {APP}/examples', shell=True, check=True)
print(json.load(open(f'{APP}/examples/labels.json')))

Start the page. Re-running this cell restarts it with the same caps.

In [ ]:
USE_MUSE = True                                   # False: spend nothing, standard instructions only
DAILY_USD, DAILY_REQUESTS, TOTAL_USD = 0.25, 60, 1.00

import threading, importlib, uvicorn
os.chdir(APP); sys.path.insert(0, APP)
import coach_core as cc, app as appmod
importlib.reload(cc); importlib.reload(appmod)
client = None
if USE_MUSE:
    from google.colab import userdata
    from openai import OpenAI
    try:
        client = OpenAI(base_url='https://api.meta.ai/v1', api_key=userdata.get('MODEL_API_KEY'), timeout=30)
    except Exception as e:
        print('No MODEL_API_KEY in Colab Secrets, so standard instructions only:', type(e).__name__)
if 'server' in globals():
    server.should_exit = True; thread.join(timeout=10)
if 'encoder' not in globals():
    encoder = cc.Encoder()
budget = appmod.Budget(client, DAILY_USD, DAILY_REQUESTS, TOTAL_USD)
coach = cc.Coach.load(MODEL)
if os.path.exists(f'{D}/quality/locator.npz'):             # measured disc and macula positions for aim directions
    coach.locator = cc.Locator.load(f'{D}/quality/locator.npz')
print('retake rule:', coach.rule, '| aim directions:', 'measured' if coach.locator else 'standard wording')
server = uvicorn.Server(uvicorn.Config(appmod.create_app(coach, encoder, budget, 'examples'),
                                       host='127.0.0.1', port=8000, log_level='warning'))
thread = threading.Thread(target=server.run, daemon=True); thread.start()

import time; time.sleep(3)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000, path='/')
print('Muse Spark wording is', 'on, capped at' if client else 'off', f'${DAILY_USD}/day, {DAILY_REQUESTS} requests/day, ${TOTAL_USD} total' if client else '')

Spending so far (the page's header shows it too):

In [ ]:
budget.status()